# 02. Сравнение моделей цены аренды

**Критерий выбора – минимальная MAE на validation.** Для подсказки арендодателю важно, на сколько рублей в среднем прогноз месячной аренды отличается от указанной в объявлении цены. MAE учитывает завышение и занижение цены одинаково. Дополнительно считаем RMSE в рублях, чтобы видеть влияние крупных промахов, и R², чтобы оценить долю объяснённой дисперсии цены.

## Подготовленные выборки

Берём train и validation из первого ноутбука. Параметры предобработки уже рассчитаны на train. Отделяем цену `rent_rub` от признаков и используем одни и те же строки validation для всех четырёх моделей.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules import RANDOM_STATE

processed_dir = PROJECT_ROOT / "data" / "processed"
train = pd.read_csv(processed_dir / "train.csv")
validation = pd.read_csv(processed_dir / "validation.csv")

TARGET = "rent_rub"
X_train, y_train = train.drop(columns=TARGET), train[TARGET]
X_validation, y_validation = validation.drop(columns=TARGET), validation[TARGET]
assert X_train.columns.equals(X_validation.columns)

print("Объявлений в train –", len(X_train))
print("Объявлений в validation –", len(X_validation))
print("Признаков –", X_train.shape[1])

Объявлений в train – 7776
Объявлений в validation – 1944
Признаков – 54


## Модели

Для первого сравнения фиксируем по одной конфигурации каждой модели.

- **DummyRegressor** предсказывает медиану аренды на train для любого объявления. Медиана даёт минимальную абсолютную ошибку среди постоянных прогнозов на train.
- **Ridge** оценивает линейную зависимость цены от признаков. Берём стандартную силу регуляризации `alpha=1`.
- **Дерево решений** может учитывать сочетания признаков. Ограничиваем глубину восемью уровнями и задаём минимум пять объявлений в листе, чтобы избежать прогнозов по единичным примерам.
- **kNN** усредняет цены десяти ближайших объявлений. Расстояния считаются по подготовленным признакам, в которых числовые столбцы уже стандартизированы.

In [2]:
models = {
    "DummyRegressor": DummyRegressor(strategy="median"),
    "Ridge": Ridge(alpha=1.0),
    "Дерево решений": DecisionTreeRegressor(
        max_depth=8, min_samples_leaf=5, random_state=RANDOM_STATE
    ),
    "kNN": KNeighborsRegressor(n_neighbors=10),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_validation)
    rows.append({
        "Модель": name,
        "MAE, ₽": mean_absolute_error(y_validation, prediction),
        "RMSE, ₽": root_mean_squared_error(y_validation, prediction),
        "R²": r2_score(y_validation, prediction),
    })

results = pd.DataFrame(rows).set_index("Модель").sort_values("MAE, ₽")
results.round({"MAE, ₽": 2, "RMSE, ₽": 2, "R²": 3})

,"MAE, ₽","RMSE, ₽",R²
Модель,,,
Ridge,4751.06,7209.58,0.784
Дерево решений,4807.55,7403.07,0.772
kNN,4873.35,7808.49,0.747
DummyRegressor,10089.25,16137.86,-0.081


При выбранных параметрах минимальная MAE на validation у Ridge – 4 751 ₽ в месяц. Для прогноза медианой train ошибка составляет 10 089 ₽. Ridge снижает её примерно на 53 %. Ближайший результат у дерева – 4 808 ₽, поэтому на этой выборке преимущество Ridge над деревом небольшое.